In [1]:
#!uv pip install llama-index-llms-huggingface llama-index-llms-huggingface-api

In [2]:
from llama_cpp.llama import Llama, LlamaGrammar
import httpx
from bs4 import BeautifulSoup
from llama_index.core.node_parser import HTMLNodeParser
import numpy as np
import pandas as pd
import torch
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import SimpleDirectoryReader, StorageContext
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.postgres import PGVectorStore
import textwrap
from llama_index.core import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from transformers import AutoTokenizer
from llama_index.core import set_global_tokenizer
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import RouterQueryEngine

/llm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/llm/.venv/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_id" in DeployedModel has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/llm/.venv/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in HuggingFaceLLM has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/llm/.venv/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_kwargs" in HuggingFaceLLM has conflict with protected namespace "model_".

You may 

In [3]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

set_global_tokenizer(
    tokenizer.encode
)

In [4]:
r = httpx.get("https://memory-alpha.fandom.com/wiki/Jean-Luc_Picard/")
html_text = r.text
soup = BeautifulSoup(html_text)
tags = [tag.name for tag in soup.find_all()]
html_doc = Document(text=html_text)

In [5]:
# The default tags are: ["p", "h1", "h2", "h3", "h4", "h5", "h6", "li", "b", "i", "u", "section"]
parser = HTMLNodeParser(tags=["p", "h1", "h2", "h3", "h4", "h5", "h6", "li", "b", "i", "u", "section"])
nodes = parser.get_nodes_from_documents([html_doc])
print(len(nodes))

24


In [6]:
import psycopg
def drop(name):
    with psycopg.connect("host=postgres dbname=grover user=grover password=grover") as conn:
        with conn.cursor() as cur:
            cur.execute(f"""
                drop table if exists {name};
                """)
            conn.commit()

drop("data_pic_html")

In [7]:
vector_store = PGVectorStore.from_params(
    database='grover',
    host='postgres',
    password='grover',
    port=5432,
    user='grover',
    table_name="pic_html",
    embed_dim=384,  
    hnsw_kwargs={
        "hnsw_m": 14,
        "hnsw_ef_construction": 72,
        "hnsw_ef_search": 52,
        "hnsw_dist_method": "vector_cosine_ops",
    },
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [8]:
index = VectorStoreIndex(nodes, storage_context=storage_context, show_progress=True)

Generating embeddings: 100%|██████████| 24/24 [00:00<00:00, 95.18it/s]


In [9]:
# llm = HuggingFaceLLM(
#     context_window=8192,
#     max_new_tokens=2048,
#     generate_kwargs={"do_sample": True, 
#                      "eos_token_id": tokenizer.eos_token_id,
#                      "top_k": 7,
#                      "top_p": 0.3,
#                      "temperature": 0.05
#                     },
#     # query_wrapper_prompt=query_wrapper_prompt,
#     tokenizer_name="meta-llama/Llama-3.1-8B-Instruct",
#     model_name="meta-llama/Llama-3.1-8B-Instruct",
#     device_map="auto",
#     tokenizer_kwargs={"max_length": 2048},
#     model_kwargs={"torch_dtype": torch.float16},
# )
# Settings.llm = llm


In [ ]:
print(index.as_query_engine(response_mode="tree_summarize").query("How did captain picard lose his heart?"))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


In [ ]:
import torch
device = torch.cuda.get_current_device()
device.reset()

In [11]:
def messages_to_prompt(messages):
    prompt = ""
    for message in messages:
        if message.role == 'system':
            prompt += f"<|system|>\n{message.content}</s>\n"
        elif message.role == 'user':
            prompt += f"<|user|>\n{message.content}</s>\n"
        elif message.role == 'assistant':
            prompt += f"<|assistant|>\n{message.content}</s>\n"

    # ensure we start with a system prompt, insert blank if needed
    if not prompt.startswith("<|system|>\n"):
        prompt = "<|system|>\n</s>\n" + prompt

    # add final assistant prompt
    prompt = prompt + "<|assistant|>\n"

    return prompt

def completion_to_prompt(completion):
    return f"<|system|>\n</s>\n<|user|>\n{completion}</s>\n<|assistant|>\n"

In [13]:
llm = LlamaCPP(
    model_url="https://huggingface.co/Orenguteng/Llama-3.1-8B-Lexi-Uncensored-V2-GGUF/blob/main/Llama-3.1-8B-Lexi-Uncensored_V2_Q8.gguf",
    context_window=16384,
    max_new_tokens=1024,
    generate_kwargs={"do_sample": False, 
                     "eos_token_id": tokenizer.eos_token_id,
                     "top_k": 1,
                     "top_p": 0.02,
                     "temperature": 0.1,
                     "repeat_penalty": 1.08
                    },
    messages_to_prompt=messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    # tokenizer_name="meta-llama/Llama-3.1-8B-Instruct",
    # tokenizer_kwargs={"max_length": 2048},
    model_kwargs={
        "n_gpu_layers": -1,
        # "grammar": grammar
                 },
    verbose = True
)
Settings.llm = llm

NameError: name 'LlamaCPP' is not defined

In [ ]:
print(index.as_query_engine(response_mode="tree_summarize").query("How did captain picard lose his heart?"))